# 지방의회 회의록: 로컬 키워드 추출과 회의별 이력

**아래 `OUTPUT_DIR`에 결과를 저장할 로컬 절대 경로를 입력하고 전체 셀을 실행하세요.**

- 입력 폴더는 요청하신 `data/format_changed_data`로 설정되어 있습니다. 그 폴더 바로 아래의 모든 `.parquet` 파일을 함께 분석하므로 2024·2025 파일을 모두 처리합니다. 하위 폴더는 탐색하지 않습니다.
- `MINTS_TEXT`에서 발언자를 분리하고, 의례적인 문장과 불용어를 정제한 뒤 Kiwi 명사·연속 명사구와 전체 회의 기준 TF-IDF로 회의별 키워드를 추출합니다.
- 회의별 상위 키워드의 합집합에 속한 단어를 모든 회의에서 다시 찾아 이력을 저장합니다. 해당 회의의 상위 키워드가 아닌 출현도 남습니다.
- 별도 Python 파일, 해시로 묶은 실행 코드, 외부 AI API가 필요하지 않습니다. 분석과 결과 저장은 이 노트북을 실행하는 **로컬 Python 환경**에서 이루어집니다. 최초 패키지 설치 시에만 인터넷이 필요할 수 있습니다.
- 원본 파일을 수정하지 않습니다. 실수로 덮어쓰지 않도록 출력 폴더는 새 폴더 또는 빈 폴더여야 합니다.

이 노트북을 작성할 때 사용자 데이터 폴더에는 접근하지 않았습니다. 예제 데이터로 기능을 검증했으므로 실제 파일에서 회의명·날짜 인식 상태와 추출 결과를 확인하세요. Python 3.10 이상을 사용합니다.


## 실행 경로와 분석 범위 설정

**입력:** 로컬 원본 폴더 `INPUT_DIR`, 결과 저장 폴더 `OUTPUT_DIR`.

**처리:** 출력 경로 입력 여부, 절대 경로 여부와 Python 버전을 점검하고 상위 개수·작업 수·표본 수를 설정합니다. `LIMIT_PER_FILE=0`은 전체 회의 처리입니다.

**결과:** 이후 셀에서 공통으로 사용하는 설정 변수가 준비됩니다. `OUTPUT_DIR`에는 새 폴더 또는 빈 폴더 경로를 입력하고, 위에서 아래로 실행하세요.

전체 흐름은 **설정 → 환경 준비 → 함수 정의 → 입력 점검 → 실제 추출·검증 → 미리보기·이력 조회 → HTML 보고서**입니다.


In [ ]:
from pathlib import Path
import os
import sys

# 입력 경로: 요청하신 경로입니다. 2024·2025 Parquet를 이 폴더에서 읽습니다.
INPUT_DIR = Path('/Users/kimhyunjin/Desktop/김현진 project/2026 데이터 AI 분석 경진대회/data/format_changed_data')

# ★ 여기만 입력하세요. 예: /Users/kimhyunjin/Desktop/keyword_results_2024_2025
OUTPUT_DIR = ""

# 선택 설정 — 기본값으로 전체 회의를 처리합니다.
TOP_KEYWORDS = 20
WORKERS = min(4, os.cpu_count() or 2)
LIMIT_PER_FILE = 0  # 0: 전체 / 2: 파일마다 앞의 2개 회의로 시험 실행

if sys.version_info < (3, 10):
    raise RuntimeError('Python 3.10 이상의 로컬 커널을 선택하세요.')
if not str(OUTPUT_DIR).strip():
    raise ValueError('OUTPUT_DIR에 결과를 저장할 로컬 절대 경로를 입력하세요.')
OUTPUT_DIR = Path(str(OUTPUT_DIR).strip()).expanduser()
if not OUTPUT_DIR.is_absolute():
    raise ValueError('OUTPUT_DIR에는 /Users/...처럼 절대 경로를 입력하세요.')
print('입력:', INPUT_DIR)
print('출력:', OUTPUT_DIR)


## 실행 환경 준비

현재 노트북 커널에 필요한 패키지가 없을 때만 설치합니다. 이미 설치된 패키지는 자동 교체하지 않습니다. 검증한 버전은 PyArrow 26.0.0, kiwipiepy 0.24.1, kiwipiepy_model 0.24.0입니다. 설치가 제한된 환경에서는 이 버전들을 미리 설치한 로컬 커널을 사용하세요.

**입력 → 처리 → 결과:** 현재 노트북 커널 → 누락 패키지 확인·설치 → 사용 가능한 패키지와 버전 출력. 설치에는 인터넷이 필요할 수 있고, 설치 셀에서는 회의록을 읽지 않습니다.


In [ ]:
import importlib
import importlib.util
import importlib.metadata
import subprocess

REQUIRED_PACKAGES = {
    'pyarrow': 'pyarrow==26.0.0',
    'kiwipiepy': 'kiwipiepy==0.24.1',
    'kiwipiepy_model': 'kiwipiepy_model==0.24.0',
}
missing = [spec for module, spec in REQUIRED_PACKAGES.items()
           if importlib.util.find_spec(module) is None]
if missing:
    print('현재 커널에 필요한 패키지를 설치합니다:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
    importlib.invalidate_caches()
for package in REQUIRED_PACKAGES:
    print(package, importlib.metadata.version(package))


## 처리 규칙과 결과 연결 방식

`MINTS_TEXT`는 필수입니다. 회의명은 `MINTS_NM`, `MTGNM`, `MTG_NM` 등에서, 의회명은 `RASMBLY_NM` 등에서, 일자는 `MTG_DATE`, `MTG_DE` 등에서 찾습니다. 메타데이터 열이 다르면 아래 전처리 규칙 셀의 `MEETING_NAME_COLUMNS`, `COUNCIL_COLUMNS`, `DATE_COLUMNS`에 열 이름을 추가할 수 있습니다. 회의명을 찾지 못하면 파일명과 행 번호를 표시하고 경고를 남깁니다.

| 연결 필드 | 의미 |
|---|---|
| `meeting_id` | `[입력 파일의 로컬 절대 경로, 0부터 시작하는 물리적 행 번호]`를 그대로 기록한 문자열 |
| `keyword_id` | NFC 정규화·영문 소문자 처리한 키워드 자체. 연도·지역이 달라도 같은 단어이면 동일 |
| `input_file`, `input_row_index` | 실제 입력 Parquet의 경로와 행 번호. 이 둘로 원본 회의를 다시 찾을 수 있음 |
| `source_file`, `source_row` | 원본에 `SOURCE_FILE`, `SOURCE_ROW`가 있으면 별도 보존 |
| `source_start`, `source_end` | 정제 전 `MINTS_TEXT`의 Python 문자 좌표. `원문[start:end]`가 근거 문장과 일치 |

파일을 옮기거나 행 순서를 바꾸면 `meeting_id`도 바뀝니다. 장기 이력은 입력 파일을 그대로 보관하면서 사용하세요. 회의명이 같거나 내용이 같아도 원본 행을 임의로 합치지 않습니다.


### 분석 라이브러리 준비

**입력:** 앞에서 준비한 Python 실행 환경과 설치 패키지.

**처리:** 경로·집계·날짜·정규표현식·JSON 기능과 PyArrow, Kiwi를 불러옵니다. 여기서는 함수를 실행하거나 회의록을 읽지 않습니다.

**결과:** 뒤쪽 셀에서 사용할 모듈과 클래스 이름이 준비됩니다. 설치 셀 다음, 모든 함수 정의보다 먼저 실행합니다.


In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date, datetime
import html
import importlib.metadata
import json
import math
import re
import unicodedata

import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.dataset as ds
import pyarrow.parquet as pq
from kiwipiepy import Kiwi


### 전처리 규칙과 발언·문장·후보어 분리

**입력:** 회의 원문, 원문 한 행, Kiwi 형태소 토큰.

**처리:** 불용어·의전 표현·화자 표식·날짜와 회의명 열 후보를 정의합니다. 아래 여섯 함수는 문자열 정규화부터 키워드 후보 추출까지의 세부 작업을 맡습니다.

| 함수 | 역할 |
|---|---|
| `normalize` | 한글의 유니코드 조합 형태를 NFC로 통일 |
| `parse_label` | 한 행에서 발언자 이름·직책·보조 정보 분리 |
| `speech_ranges` | 화자별 발언 구간과 머리말·꼬리말 분리 |
| `chunks` | 긴 발언을 원문 좌표가 유지되는 조각으로 분할 |
| `clean_sentence` | 순수 의전·진행 문장 제외와 문장 앞 인사말 정제 |
| `noun_candidates` | 명사·외국어와 연속 명사구 후보 생성 |

**결과:** 이후 `preprocess`가 호출할 규칙과 함수가 정의됩니다. 이 셀 자체는 분석 파일을 만들지 않습니다. `chunks`는 겹치지 않는 분할이며, 제외 토큰을 사이에 둔 명사끼리는 후보 명사구로 연결하지 않습니다.


In [ ]:
STOPWORDS = set('''것 수 등 및 때 이 그 저 우리 여러분 제가 저희 이것 그것 저것
정도 부분 경우 말씀 답변 질문 질의 보고 설명 생각 사항 내용 관련 오늘 지금 현재 다음
위원 의원 위원장 부위원장 의장 부의장 시장 도지사 교육감 국장 과장 실장 처장 원장
팀장 담당관 전문위원 수석전문위원 사무처장 본부장 사무관 차관 장관 부시장 부지사
회의 의회 회의록 개의 산회 정회 속개 수고 감사하다 감사합니다 부탁 말씀드리다
개의선언 자료화면 화면자료 페이지 쪽수 성원 재석 가결 부결 원안 표결 호선 의사일정
top hwp pdf html http https www 다운로드 창닫기 파일 첨부'''.split())
# '감사'는 행정감사에도 쓰이므로 단어 전체를 불용어로 지정하지 않습니다.
STOPWORDS.discard('감사')
STOP_PHRASES = {'자료 화면', '화면 자료', '제안 설명', '질의 답변', '의사 일정'}
MAX_NGRAM = 3
MAX_CHUNK_CHARS = 6000
# 실제 파일의 메타데이터 열 이름이 다르면 후보에 추가할 수 있습니다.
MEETING_NAME_COLUMNS = ('MINTS_NM', 'MTGNM', 'MTG_NM', 'MTG_SJ', 'MEETING_NAME', '회의명')
COUNCIL_COLUMNS = ('RASMBLY_NM', 'COUNCIL_NAME', 'COUNCIL', '의회명')
DATE_COLUMNS = ('MTG_DATE', 'MTG_DE', 'MINTS_DATE', 'MEETING_DATE', '회의일자')

TITLE_END = r'(?:수석전문위원|부위원장|위원장|부의장|의장|전문위원|의원|위원|부시장|시장|부지사|도지사|교육감|교육장|사무처장|담당관|기획관|정책관|본부장|국장|과장|실장|처장|원장|팀장|사무관|대표이사|이사장)'
ROLE = r'(?:[가-힣A-Za-z0-9·.\-/]{0,25}' + TITLE_END + r'(?:[ \t]*(?:직무대리|권한대행))?)'
NAME = r'(?P<name>(?:[가-힣](?:[ \t]+[가-힣]){1,3}|[가-힣]{2,4}))'
QUALIFIER = r'(?:[ \t]*(?P<qualifier>[（(][^()（）\n]{1,30}[）)]))?'
LABELS = [
    re.compile(r'(?P<role>' + ROLE + r')[ \t]*' + NAME + QUALIFIER + r'(?=[ \t:：]|$)'),
    re.compile(NAME + QUALIFIER + r'[ \t]*(?P<role>' + ROLE + r')(?=[ \t:：]|$)'),
]
MARKER = re.compile(r'^[ \t]*[○◯●◎▶▷][ \t]*')
END_MARKER = re.compile(r'(?m)^[ \t]*[（(]\s*\d{1,2}시\s*\d{0,2}분?\s*(?:산회|폐회)\s*[）)][ \t]*$')
GREETING_PREFIX = re.compile(r'^(?:안녕하십니까|안녕하세요|감사합니다|수고하셨습니다|답변드리겠습니다)[.!?。\s,，]+')
SALUTATION = re.compile(r'^존경하는\s+[가-힣·\s]{1,35}(?:위원장님|의장님|의원님|위원님|여러분)[!,.。\s]+')
PURE_NOISE = [
    re.compile(r'^(?:네|예|아니요|감사합니다|감사드립니다|고맙습니다|수고하셨습니다|수고하셨습니다만|알겠습니다|이상입니다|이상으로 마치겠습니다|답변드리겠습니다|질의하겠습니다)[.!?。\s]*$'),
    re.compile(r'^(?:성원이\s*되었으므로\s*)?(?:제\s*\d+회.{0,60}?)?(?:개의|산회|정회|속개|폐회)(?:를|을)?\s*(?:선언합니다|선포합니다|하겠습니다|합니다)[.!。\s]*$'),
    re.compile(r'^(?:다음|계속)(?:으로|은)?\s*(?:[가-힣]{2,4}\s*)?(?:위원님|의원님|국장님|과장님)?\s*(?:질의|답변|설명)(?:해|하여)?\s*주시기\s*바랍니다[.!。\s]*$'),
    re.compile(r'^(?:안녕하십니까[.!\s]*)?(?:저는\s*)?' + ROLE + r'\s*[가-힣]{2,4}입니다[.!。\s]*$'),
    re.compile(r'^[（(]\s*(?:박수|웃음|장내\s*소란|의사봉\s*\d+타|자료\s*화면을\s*보며|영상\s*자료를\s*보며|\d{1,2}시\s*\d{0,2}분?\s*(?:개의|정회|속개|산회|폐회))\s*[）)]$'),
]


# 기능: 값을 문자열로 바꾸고 한글 조합 형태를 NFC로 통일합니다.
# 입력: 문자열·숫자 등 임의의 값. None은 빈 문자열로 처리합니다.
# 반환: 정규화한 문자열. 영문 소문자화는 이 함수에서 하지 않습니다.
def normalize(value):
    return unicodedata.normalize('NFC', '' if value is None else str(value))


# 기능: 한 행의 화자 표식 또는 이름·직책만 있는 독립 행에서 발언자를 식별합니다.
# 입력: 원문 한 행. 직책, 성명, 괄호 안 선거구 등의 보조 정보를 분리합니다.
# 반환: (행 내부 본문 시작 위치, 이름, 직책, 보조 정보). 표식만 알면 이름 등은 None, 경계도 아니면 None입니다.
def parse_label(line):
    """명확한 화자 표식 또는 화자만 적힌 독립 행만 인식합니다."""
    marker = MARKER.match(line)
    offset = marker.end() if marker else len(line) - len(line.lstrip(' \t'))
    content = line[offset:]
    for pattern in LABELS:
        match = pattern.match(content)
        if not match:
            continue
        remainder = content[match.end():]
        if not marker and remainder.strip(' \t:：\r\n'):
            continue
        name = re.sub(r'\s+', '', match['name'])
        if name in {'직무대리', '권한대행'}:
            continue
        qualifier = (match['qualifier'] or '').strip('()（）')
        role = match['role']
        end = offset + match.end()
        while end < len(line) and line[end] in ' \t:：':
            end += 1
        return end, normalize(name), normalize(role), normalize(qualifier)
    # 미인식 화자 표식도 경계로 남겨 직전 화자로 잘못 연결하지 않습니다.
    if marker:
        return marker.end(), None, None, None
    return None


# 기능: 회의록을 화자별 발언 구간으로 나누고 첫 화자 앞 머리말과 최종 산회 뒤 꼬리말을 분리합니다.
# 입력: 회의 한 건의 MINTS_TEXT 원문. 화자 경계가 없으면 본문을 미상 화자의 구간으로 취급합니다.
# 반환: (발언 구간 목록, 머리말, 꼬리말). 각 구간은 원문 [시작:끝] 좌표와 화자 정보를 담습니다.
def speech_ranges(text):
    labels = []
    offset = 0
    for line in text.splitlines(keepends=True):
        found = parse_label(line)
        if found:
            local_end, name, role, qualifier = found
            labels.append((offset, offset + local_end, name, role, qualifier))
        offset += len(line)
    endings = list(END_MARKER.finditer(text))
    body_end = endings[-1].start() if endings else len(text)
    if labels:
        labels = [label for label in labels if label[0] < body_end]
    ranges = []
    if labels:
        header = text[:labels[0][0]]
        for i, label in enumerate(labels):
            _, start, name, role, qualifier = label
            end = labels[i + 1][0] if i + 1 < len(labels) else body_end
            if start < end:
                ranges.append((start, end, name, role, qualifier))
    else:
        header = ''
        if body_end:
            ranges.append((0, body_end, None, None, None))
    return ranges, header, text[body_end:]


# 기능: 긴 발언을 MAX_CHUNK_CHARS 이하의 겹치지 않는 조각으로 순차 분할합니다.
# 입력: 원문과 발언의 시작·끝 좌표. 가능하면 조각 후반의 줄바꿈 또는 공백에서 나눕니다.
# 산출: (원문 기준 조각 시작 위치, 조각 문자열)을 하나씩 yield합니다. 문장 경계가 보장되지는 않습니다.
def chunks(text, start, end):
    while start < end:
        right = min(start + MAX_CHUNK_CHARS, end)
        if right < end:
            boundary = text.rfind('\n', start + MAX_CHUNK_CHARS // 2, right)
            if boundary < 0:
                boundary = text.rfind(' ', start + MAX_CHUNK_CHARS // 2, right)
            if boundary >= 0:
                right = boundary + 1
        yield start, text[start:right]
        start = right


# 기능: 순수 인사·진행 문장은 제외하고, 정책 내용이 남는 문장은 앞부분의 인사 표현만 정제합니다.
# 입력: 정제 전 문장 문자열. 미리 정의한 정규표현식에 일치하는 표현만 처리합니다.
# 반환: (정제 문장, 처리 사유). 제외된 문장은 빈 문자열이며, 변경이 없으면 사유도 빈 문자열입니다.
def clean_sentence(raw):
    text = raw.strip()
    if not text:
        return '', 'blank'
    if any(pattern.fullmatch(text) for pattern in PURE_NOISE):
        return '', 'ceremonial_or_procedural'
    clean = SALUTATION.sub('', text, count=1)
    clean = GREETING_PREFIX.sub('', clean, count=1).strip()
    if not clean or any(pattern.fullmatch(clean) for pattern in PURE_NOISE):
        return '', 'ceremonial_or_procedural'
    return clean, 'ceremonial_prefix_removed' if clean != text else ''


# 기능: Kiwi 토큰에서 일반명사·고유명사·외국어와 연속된 1~3개 후보의 명사구를 추출합니다.
# 입력: 형태소 토큰 목록. 두 글자 미만·숫자·불용어를 제외하고 제외 토큰에서 연결을 끊습니다.
# 반환: 후보 문자열 목록. 빈도 집계를 위해 반복 출현을 유지하며, 후보는 NFC·영문 소문자로 통일합니다.
def noun_candidates(tokens):
    run = []
    result = []
    for token in tokens:
        word = normalize(token.form).lower().strip()
        if token.tag not in ('NNG', 'NNP', 'SL') or len(word) < 2 or word in STOPWORDS or word.isdigit():
            run = []
            continue
        result.append(word)
        run = (run + [word])[-MAX_NGRAM:]
        for length in range(2, min(MAX_NGRAM, len(run)) + 1):
            phrase = ' '.join(run[-length:])
            if len(set(run[-length:])) > 1 and phrase not in STOP_PHRASES:
                result.append(phrase)
    return result


### 결과 자료 구조와 파일 입출력 도구

**입력:** PyArrow 스키마 정의, 입력 경로, 회의 행의 메타데이터.

**처리:** 회의·문장·후보 빈도·키워드·이력·근거·집계 Parquet의 열 이름과 자료형을 정의합니다. `TableWriter`는 파일을 열고 행을 이어 쓴 뒤 닫는 역할을 맡습니다.

| 함수·메서드 | 역할 |
|---|---|
| `TableWriter.__init__ / add / close` | ZSTD 압축 Parquet 생성·행 기록·기록 종료 |
| `iter_parquet` | 필요한 열을 배치로 읽고 한 행씩 제공 |
| `get_date / first_nonempty` | 날짜 변환과 메타데이터 대체 값 선택 |
| `inspect_inputs` | 입력 경로·필수 열·본문 자료형·행 수 점검 |
| `write_json` | 설정·상태·검사 결과를 JSON으로 저장 |

**결과:** 실제 처리에서 사용하는 스키마와 입출력 함수가 준비됩니다. 이 정의 셀에서는 파일을 생성하지 않으며, 각 함수를 호출할 때 읽기·쓰기가 발생합니다.


In [ ]:
COMMON = [('meeting_id', pa.string()), ('meeting_name', pa.string()), ('meeting_date', pa.string()),
          ('meeting_year', pa.int64()), ('source_year', pa.int64()), ('council', pa.string()),
          ('input_file', pa.string()), ('input_row_index', pa.int64()),
          ('source_file', pa.string()), ('source_row', pa.string())]
MEETINGS_SCHEMA = pa.schema(COMMON + [('raw_chars', pa.int64()), ('kept_sentences', pa.int64()),
                                     ('removed_sentences', pa.int64()), ('recognized_speakers', pa.int64()),
                                     ('header_text', pa.string()), ('tail_text', pa.string()), ('warnings', pa.list_(pa.string()))])
SENTENCE_SCHEMA = pa.schema([('sentence_id', pa.string()), ('meeting_id', pa.string()),
                            ('speaker_name', pa.string()), ('speaker_role', pa.string()), ('speaker_qualifier', pa.string()),
                            ('source_start', pa.int64()), ('source_end', pa.int64()), ('sentence_raw', pa.string()),
                            ('sentence_text', pa.string()), ('kept', pa.bool_()), ('removal_reason', pa.string()),
                            ('terms', pa.list_(pa.string()))])
TERMS_SCHEMA = pa.schema([('meeting_id', pa.string()), ('terms', pa.list_(pa.string())), ('counts', pa.list_(pa.int64()))])
KEYWORDS_SCHEMA = pa.schema(COMMON + [('keyword_id', pa.string()), ('keyword', pa.string()), ('rank', pa.int64()),
                                     ('frequency', pa.int64()), ('document_frequency', pa.int64()), ('score', pa.float64())])
HISTORY_SCHEMA = pa.schema(COMMON + [('keyword_id', pa.string()), ('keyword', pa.string()), ('frequency', pa.int64()),
                                    ('score', pa.float64()), ('is_top_keyword', pa.bool_()), ('keyword_rank', pa.int64())])
EVIDENCE_SCHEMA = pa.schema(COMMON + [('keyword_id', pa.string()), ('keyword', pa.string()), ('sentence_id', pa.string()),
                                     ('speaker_name', pa.string()), ('speaker_role', pa.string()),
                                     ('source_start', pa.int64()), ('source_end', pa.int64()),
                                     ('sentence_raw', pa.string()), ('sentence_text', pa.string()), ('evidence_rank', pa.int64())])
CATALOG_SCHEMA = pa.schema([('keyword_id', pa.string()), ('keyword', pa.string()), ('meeting_count', pa.int64()),
                           ('total_frequency', pa.int64()), ('first_date', pa.string()), ('last_date', pa.string()),
                           ('councils', pa.list_(pa.string()))])


class TableWriter:
    # 기능: 지정한 스키마와 ZSTD 압축으로 Parquet 작성기를 열고 누적 행 수를 초기화합니다.
    # 입력: 저장 파일 경로 path와 PyArrow 스키마 schema.
    # 효과: self.writer·self.schema·self.count를 준비합니다. 사용 후 close()로 파일을 마무리해야 합니다.
    def __init__(self, path, schema):
        self.schema = schema
        self.writer = pq.ParquetWriter(path, schema, compression='zstd')
        self.count = 0

    # 기능: 딕셔너리 행 목록을 스키마에 맞는 Arrow Table로 바꾸어 현재 Parquet에 이어 씁니다.
    # 입력: 저장할 rows 목록. 빈 목록이면 기록하지 않습니다.
    # 효과: 파일에 행을 기록하고 self.count를 늘립니다. 별도 반환값은 없습니다.
    def add(self, rows):
        if rows:
            self.writer.write_table(pa.Table.from_pylist(rows, schema=self.schema))
            self.count += len(rows)

    # 기능: Parquet 작성기를 닫아 파일 기록을 마무리하고 자원을 해제합니다.
    # 입력: TableWriter 인스턴스의 현재 작성기.
    # 효과: writer.close()를 호출합니다. 별도 반환값은 없습니다.
    def close(self):
        self.writer.close()


# 기능: Parquet를 배치로 읽고 각 행을 딕셔너리로 순차 제공합니다.
# 입력: 파일 경로, 선택 열(columns=None이면 전체), 배치 행 수.
# 산출: 행 딕셔너리를 하나씩 yield합니다. 읽기 종료 시 파일을 닫습니다.
def iter_parquet(path, columns=None, batch_size=16):
    with pq.ParquetFile(path) as file:
        for batch in file.iter_batches(batch_size=batch_size, columns=columns):
            yield from batch.to_pylist()


# 기능: DATE_COLUMNS 순서로 회의 날짜를 찾아 ISO 날짜 문자열과 연도로 변환합니다.
# 입력: 회의 행 딕셔너리. date/datetime, YYYYMMDD, ISO 날짜로 시작하는 문자열을 처리합니다.
# 반환: (YYYY-MM-DD 또는 None, 연도 또는 None, 날짜 파싱 오류 경험 여부). 결측만 있으면 오류는 False입니다.
def get_date(row):
    invalid = False
    for field in DATE_COLUMNS:
        value = row.get(field)
        if value is None or str(value).strip() == '':
            continue
        try:
            if isinstance(value, datetime):
                day = value.date()
            elif isinstance(value, date):
                day = value
            else:
                value = str(value).strip()
                day = datetime.strptime(value, '%Y%m%d').date() if re.fullmatch(r'\d{8}', value) else date.fromisoformat(value[:10])
            return day.isoformat(), day.year, invalid
        except (ValueError, TypeError, OverflowError):
            invalid = True
    return None, None, invalid


# 기능: 여러 후보 중 None이 아니고 문자열 표현이 공백뿐이지 않은 첫 값을 선택합니다.
# 입력: 우선순위대로 나열한 값들. 숫자 0도 유효한 값입니다.
# 반환: 처음 찾은 원래 값이며, 유효한 후보가 없으면 None입니다.
def first_nonempty(*values):
    return next((value for value in values if value is not None and str(value).strip()), None)


# 기능: 입력 폴더 바로 아래 Parquet의 존재 여부, MINTS_TEXT 열과 자료형, 전체 행 수를 점검합니다.
# 입력: 존재하는 로컬 절대 폴더 경로. 하위 폴더를 재귀 탐색하지 않습니다.
# 반환: 파일 경로·행 수·크기·수정 시각 목록. 입력이 부적절하거나 모든 파일이 0행이면 예외를 발생시킵니다.
def inspect_inputs(input_dir):
    input_dir = Path(input_dir)
    if not input_dir.is_absolute() or not input_dir.is_dir():
        raise ValueError(f'입력은 존재하는 로컬 절대 경로여야 합니다: {input_dir}')
    files = sorted((p for p in input_dir.iterdir() if p.is_file() and p.suffix.lower() == '.parquet'),
                   key=lambda p: normalize(p.name))
    if not files:
        raise FileNotFoundError(f'Parquet 파일이 없습니다: {input_dir}')
    inventory = []
    for path in files:
        with pq.ParquetFile(path) as pf:
            if 'MINTS_TEXT' not in pf.schema_arrow.names:
                raise ValueError(f'{path.name}: MINTS_TEXT 컬럼이 없습니다.')
            dtype = pf.schema_arrow.field('MINTS_TEXT').type
            if pa.types.is_dictionary(dtype):
                dtype = dtype.value_type
            if not (pa.types.is_string(dtype) or pa.types.is_large_string(dtype) or pa.types.is_null(dtype)):
                raise TypeError(f'{path.name}: MINTS_TEXT는 문자열이어야 합니다. 현재 {dtype}')
            inventory.append({'path': str(path.absolute()), 'rows': pf.metadata.num_rows,
                              'size_bytes': path.stat().st_size, 'mtime_ns': path.stat().st_mtime_ns})
    if not sum(item['rows'] for item in inventory):
        raise ValueError('모든 입력 파일이 0행입니다.')
    return inventory


# 기능: 실행 설정·상태·검증 결과 등을 읽기 쉬운 UTF-8 JSON으로 저장합니다.
# 입력: 저장 경로 path와 직렬화할 객체 obj. 기본 JSON 변환이 안 되는 값은 문자열로 기록합니다.
# 효과: 지정 파일에 내용을 씁니다. 상위 폴더 생성이나 기존 파일 보호는 호출 측에서 처리합니다.
def write_json(path, obj):
    Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')


### 회의별 전처리와 추적 가능한 문장 데이터 구축

**입력:** `inspect_inputs`가 만든 파일 목록, 출력 경로, 작업 수, 파일별 표본 수.

**처리:** `preprocess`가 회의를 배치로 읽고 원문 경로·물리적 행 번호로 회의 ID를 만듭니다. 화자와 문장을 분리한 뒤 의전 표현을 정제하고 후보 빈도를 집계합니다. 문장마다 정제 전 원문 좌표가 맞는지 직접 대조하며, 빈 본문·누락 날짜·미인식 화자는 경고로 남깁니다.

**호출 시 저장 결과:** `meetings.parquet`(회의 메타정보), `sentences.parquet`(문장·화자·원문 좌표), `term_counts.parquet`(후보 빈도).

**반환값:** 회의 메타정보, 의회별 화자 이름, 직책 목록, 정제·좌표 검사 통계. 이 셀은 함수를 정의하며 실제 실행은 아래의 실행 셀에서 시작합니다.


In [ ]:
# 기능: 회의를 배치로 읽어 발언자 분리→문장 분할·정제→후보어 빈도 집계를 수행하고 원문 좌표를 대조합니다.
# 입력: 입력 파일 목록 inventory, 출력 폴더 out, Kiwi 작업 수, 파일별 표본 수(0이면 전체).
# 저장: meetings.parquet, sentences.parquet, term_counts.parquet. 결측 본문 회의도 메타정보에 보존합니다.
# 반환: (회의 메타정보 사전, 의회별 화자 이름 집합, 직책 집합, 좌표 검사·제외 문장 통계).
def preprocess(inventory, out, workers=4, limit_per_file=0):
    kiwi = Kiwi(num_workers=workers, model_type='cong')
    writers = {
        'meetings': TableWriter(out / 'meetings.parquet', MEETINGS_SCHEMA),
        'sentences': TableWriter(out / 'sentences.parquet', SENTENCE_SCHEMA),
        'terms': TableWriter(out / 'term_counts.parquet', TERMS_SCHEMA),
    }
    meetings = {}
    people = defaultdict(set)
    roles = set()
    slice_checks = 0
    removed_count = 0
    try:
        for info in inventory:
            path = Path(info['path'])
            source_year_match = re.match(r'^(\d{4})_', path.name)
            source_year = int(source_year_match[1]) if source_year_match else None
            with pq.ParquetFile(path) as file:
                columns = [name for name in file.schema_arrow.names if name != 'MINTS_HTML']
            for row_index, row in enumerate(iter_parquet(path, columns, batch_size=8)):
                if limit_per_file and row_index >= limit_per_file:
                    break
                raw = row.get('MINTS_TEXT') or ''
                # 해시 대신 파일 경로와 0부터 시작하는 물리 행 번호를 그대로 사용합니다.
                mid = json.dumps([str(path.absolute()), row_index], ensure_ascii=False, separators=(',', ':'))
                if mid in meetings:
                    raise ValueError(f'중복 회의 식별자: {mid}')
                fallback_council = re.sub(r'^\d{4}_', '', path.stem)
                council = normalize(first_nonempty(*(row.get(field) for field in COUNCIL_COLUMNS), fallback_council))
                source_meeting_name = first_nonempty(*(row.get(field) for field in MEETING_NAME_COLUMNS))
                meeting_name = normalize(first_nonempty(source_meeting_name, f'{path.stem} / 행 {row_index}'))
                meeting_date, meeting_year, invalid_date = get_date(row)
                source_file = str(first_nonempty(row.get('SOURCE_FILE'), path.name))
                source_row = str(row['SOURCE_ROW']) if row.get('SOURCE_ROW') is not None else None
                common = dict(meeting_id=mid, meeting_name=meeting_name, meeting_date=meeting_date,
                              meeting_year=meeting_year, source_year=source_year, council=council,
                              input_file=str(path.absolute()), input_row_index=row_index,
                              source_file=source_file, source_row=source_row)
                warnings = []
                if source_meeting_name is None: warnings.append('missing_meeting_name_using_file_and_row')
                if not raw.strip(): warnings.append('empty_text')
                if not meeting_date: warnings.append('missing_meeting_date')
                if invalid_date: warnings.append('invalid_date_value')
                if meeting_year is not None and source_year is not None and meeting_year != source_year:
                    warnings.append('meeting_year_source_year_mismatch')
                ranges, header, tail = speech_ranges(raw)
                if ranges and not any(name for _, _, name, _, _ in ranges):
                    warnings.append('speaker_not_identified')
                if any(not name for _, _, name, _, _ in ranges):
                    warnings.append('unidentified_speaker_present')
                for _, _, name, role, _ in ranges:
                    if name: people[council].add(name)
                    if role: roles.add(role.replace(' ', ''))
                counter = Counter()
                sentence_rows = []
                kept = removed = 0
                for speech_start, speech_end, name, role, qualifier in ranges:
                    for offset, chunk in chunks(raw, speech_start, speech_end):
                        for sentence in kiwi.split_into_sents(chunk, return_tokens=True):
                            start = offset + sentence.start
                            end = offset + sentence.end
                            sentence_raw = chunk[sentence.start:sentence.end]
                            assert raw[start:end] == sentence_raw, '원문 문자 좌표 불일치'
                            slice_checks += 1
                            cleaned, reason = clean_sentence(sentence_raw)
                            tokens = sentence.tokens if cleaned == sentence_raw.strip() else (kiwi.tokenize(cleaned) if cleaned else [])
                            terms = noun_candidates(tokens) if cleaned else []
                            if cleaned:
                                counter.update(terms)
                                kept += 1
                            else:
                                removed += 1
                            sentence_rows.append(dict(sentence_id=f'{mid}::chars={start}:{end}', meeting_id=mid,
                                speaker_name=name, speaker_role=role, speaker_qualifier=qualifier,
                                source_start=start, source_end=end, sentence_raw=sentence_raw,
                                sentence_text=cleaned, kept=bool(cleaned), removal_reason=reason,
                                terms=list(dict.fromkeys(terms))))
                            if len(sentence_rows) >= 1000:
                                writers['sentences'].add(sentence_rows)
                                sentence_rows = []
                writers['sentences'].add(sentence_rows)
                removed_count += removed
                if not counter: warnings.append('no_keyword_candidates')
                summary = dict(common, raw_chars=len(raw), kept_sentences=kept, removed_sentences=removed,
                               recognized_speakers=len({name for _, _, name, _, _ in ranges if name}),
                               header_text=header, tail_text=tail, warnings=warnings)
                writers['meetings'].add([summary])
                writers['terms'].add([dict(meeting_id=mid, terms=list(counter), counts=list(counter.values()))])
                meetings[mid] = common
                if len(meetings) % 50 == 0:
                    print(f'전처리 {len(meetings):,}개 회의 완료', flush=True)
            print(f'완료: {path.name} / 누적 {len(meetings):,}개 회의', flush=True)
    finally:
        for writer in writers.values(): writer.close()
    return meetings, people, roles, {'source_slice_checks': slice_checks, 'removed_sentences': removed_count}


### 공통 TF-IDF로 키워드 선정 및 전체 출현 이력 연결

**입력:** 전처리 Parquet와 회의·화자·직책 정보, 상위 키워드 개수.

**처리 순서:**

1. 내부 함수 `eligible`로 의회명·발언자·직책·불용어 후보를 걸러냅니다.
2. 전체 회의에서 단어별 문서빈도를 구하고 `score_terms`로 `(1 + ln 빈도) × [ln((1 + 회의 수)/(1 + 문서빈도)) + 1]`을 계산한 뒤 회의별 L2 정규화를 적용합니다.
3. 회의별 상위 키워드를 선택합니다. 먼저 선택된 명사구 안에 포함되는 짧은 후보는 건너뜁니다.
4. 모든 회의의 상위 키워드 합집합을 추적 대상으로 삼아, 다른 회의의 비상위 출현까지 이력에 기록합니다.
5. 키워드 집계를 만들고, 회의별 상위 키워드마다 먼저 등장한 근거 문장 최대 3개를 연결합니다.

**호출 시 저장 결과:** `meeting_keywords.parquet`, `keyword_history.parquet`, `keyword_catalog.parquet`, `keyword_evidence.parquet`, `exclusion_lists.json`.

**반환값:** 처리·추출 규모 통계. TF-IDF 점수는 중요도의 절대값이나 확률이 아니며, 근거 문장은 생성형 요약문이 아닙니다.


In [ ]:
# 기능: 전처리 후보에서 화자·직책 등을 제외하고 공통 TF-IDF로 회의별 상위 키워드를 선정합니다.
# 입력: 전처리 결과 폴더, 회의 메타정보·화자·직책 목록, 회의별 최대 키워드 수 top_k.
# 저장: 회의별 키워드, 선정 단어의 전체 회의 출현 이력, 키워드 집계, 첫 근거 문장 최대 3개와 제외 목록.
# 반환: 처리 회의 수·키워드 수·이력 행 수 등 통계. 다른 회의의 비상위 출현도 추적 대상에 포함합니다.
def extract_keywords(out, meetings, people, roles, top_k=20):
    N = len(meetings)
    write_json(out / 'exclusion_lists.json', {
        'speaker_names_by_council': {council: sorted(names) for council, names in people.items()},
        'speaker_roles': sorted(roles),
    })
    # 기능: 현재 회의에서 후보가 불용구·직책·의회명·화자 이름 제외 규칙을 통과하는지 판단합니다.
    # 입력: 후보 문자열 word와 회의 ID mid. 바깥 함수의 meetings·people·roles를 참조합니다.
    # 반환: 추출 가능하면 True, 제외 대상이면 False입니다.
    def eligible(word, mid):
        names = people[meetings[mid]['council']]
        if word in STOP_PHRASES or word.replace(' ', '') in roles:
            return False
        council_names = {meetings[mid]['council'], meetings[mid]['council'].removesuffix('의회')}
        return word not in council_names and not any(part in STOPWORDS or part in names for part in word.split())

    df = Counter()
    for row in iter_parquet(out / 'term_counts.parquet'):
        df.update(word for word in row['terms'] if eligible(word, row['meeting_id']))
    idf = {word: math.log((1 + N) / (1 + count)) + 1 for word, count in df.items()}

    # 기능: 회의별 후보 빈도에 로그 TF와 공통 IDF를 적용하고 L2 길이로 정규화합니다.
    # 입력: term_counts의 한 회의 행. 바깥 함수에서 계산한 IDF와 eligible 규칙을 사용합니다.
    # 반환: {후보 문자열: 정규화 TF-IDF 점수} 사전. 유효한 후보가 없으면 빈 사전입니다.
    def score_terms(row):
        mid = row['meeting_id']
        scores = {word: (1 + math.log(freq)) * idf[word]
                  for word, freq in zip(row['terms'], row['counts']) if word in idf and eligible(word, mid)}
        denominator = math.sqrt(sum(score * score for score in scores.values())) or 1.0
        return {word: score / denominator for word, score in scores.items()}

    selections = {}
    vocab = set()
    kw_writer = TableWriter(out / 'meeting_keywords.parquet', KEYWORDS_SCHEMA)
    try:
        for row in iter_parquet(out / 'term_counts.parquet'):
            mid = row['meeting_id']
            scores = score_terms(row)
            selected = []
            for word, score in sorted(scores.items(), key=lambda item: (-item[1], item[0])):
                if any((' ' + word + ' ') in (' ' + chosen + ' ') for chosen, _ in selected):
                    continue
                selected.append((word, score))
                if len(selected) == top_k: break
            counts = dict(zip(row['terms'], row['counts']))
            rows = [dict(meetings[mid], keyword_id=word, keyword=word, rank=rank,
                         frequency=counts[word], document_frequency=df[word], score=score)
                    for rank, (word, score) in enumerate(selected, 1)]
            kw_writer.add(rows)
            selections[mid] = {r['keyword']: r for r in rows}
            vocab.update(selections[mid])
    finally:
        kw_writer.close()

    history_writer = TableWriter(out / 'keyword_history.parquet', HISTORY_SCHEMA)
    catalog = {}
    try:
        for row in iter_parquet(out / 'term_counts.parquet'):
            mid = row['meeting_id']
            scores = score_terms(row)
            records = []
            for word, frequency in zip(row['terms'], row['counts']):
                if word not in vocab or word not in scores: continue
                top = selections[mid].get(word)
                records.append(dict(meetings[mid], keyword_id=word, keyword=word, frequency=frequency,
                                    score=scores[word], is_top_keyword=top is not None,
                                    keyword_rank=top['rank'] if top else None))
                entry = catalog.setdefault(word, dict(keyword_id=word, keyword=word, meeting_count=0,
                    total_frequency=0, first_date=None, last_date=None, councils=set()))
                entry['meeting_count'] += 1
                entry['total_frequency'] += frequency
                entry['councils'].add(meetings[mid]['council'])
                day = meetings[mid]['meeting_date']
                if day:
                    entry['first_date'] = min(day, entry['first_date'] or day)
                    entry['last_date'] = max(day, entry['last_date'] or day)
            history_writer.add(records)
    finally:
        history_writer.close()
    catalog_writer = TableWriter(out / 'keyword_catalog.parquet', CATALOG_SCHEMA)
    try:
        catalog_writer.add([dict(entry, councils=sorted(entry['councils'])) for entry in catalog.values()])
    finally:
        catalog_writer.close()

    # 각 회의의 상위 키워드마다 먼저 등장한 근거 문장 3개를 보존합니다.
    evidence_writer = TableWriter(out / 'keyword_evidence.parquet', EVIDENCE_SCHEMA)
    current = None
    evidence_counts = Counter()
    try:
        for row in iter_parquet(out / 'sentences.parquet', batch_size=2048):
            mid = row['meeting_id']
            if mid != current:
                current = mid
                evidence_counts.clear()
            if not row['kept']: continue
            records = []
            for word in row['terms']:
                if word not in selections[mid] or evidence_counts[word] >= 3: continue
                evidence_counts[word] += 1
                records.append(dict(meetings[mid], keyword_id=word, keyword=word,
                    sentence_id=row['sentence_id'], speaker_name=row['speaker_name'], speaker_role=row['speaker_role'],
                    source_start=row['source_start'], source_end=row['source_end'],
                    sentence_raw=row['sentence_raw'], sentence_text=row['sentence_text'], evidence_rank=evidence_counts[word]))
            evidence_writer.add(records)
    finally:
        evidence_writer.close()
    return dict(meetings=N, keywords=kw_writer.count, history_rows=history_writer.count,
                tracked_keywords=len(vocab), evidence_rows=evidence_writer.count,
                meetings_without_keywords=sum(not value for value in selections.values()))


### 무결성 검사·전체 처리 실행·이력 조회 함수

**입력:** 원본·출력 폴더와 실행 설정, 또는 이미 생성된 결과 폴더와 검색어.

| 함수 | 역할과 결과 |
|---|---|
| `validate_outputs` | 후보 빈도에서 점수·순위·전체 이력을 재계산하는 등 15개 일관성 검사. `validation.json` 저장 후 실패 시 예외 발생 |
| 내부 `check / valid_candidate` | 검사 결과 수집과 후보 적격성 재판정 |
| `run_local_extraction` | 경로·빈 출력 폴더 확인 → 전처리 → 추출 → 검증 → 실행 명세 저장 |
| `keyword_history` | 저장된 이력에서 검색어·선택 연도로 조회하고 날짜·의회 순으로 정렬 |

**호출 시 상태 기록:** 처리 시작 시 `RUNNING.json`, 정상 완료 시 `run_manifest.json`을 남기고 `RUNNING.json`을 삭제합니다. 처리 단계 예외는 `FAILED.json`에 기록합니다. HTML 보고서는 마지막 별도 셀에서 생성합니다.

**실행 순서:** 여기까지는 함수 정의입니다. 다음 입력 확인 셀과 전체 실행 셀에서 실제 파일 작업을 시작합니다. 자동 검증은 데이터 일관성을 확인하며 의미적 정확도를 보장하지는 않습니다.


In [ ]:
# 기능: 생성된 Parquet의 회의 수·ID·메타정보·빈도·점수·순위·전체 이력·근거 연결을 15개 항목으로 검사합니다.
# 입력: 결과 폴더 out, 예상 회의 수 expected_meetings, 회의별 상위 개수 top_k.
# 저장/반환: validation.json을 기록하고 검사 결과 사전을 반환합니다. 실패 항목이 있으면 AssertionError를 발생시킵니다.
# 범위: 데이터 일관성을 확인하는 검사이며 키워드의 의미적 적절성까지 판단하지는 않습니다.
def validate_outputs(out, expected_meetings, top_k):
    metadata = {r['meeting_id']: r for r in iter_parquet(out / 'meetings.parquet')}
    checks = []
    # 기능: 검증 항목 하나의 이름·통과 여부·상세 정보를 결과 목록에 추가합니다.
    # 입력: 검사명 name, 참/거짓 passed, 설명 또는 수치 detail.
    # 효과: 바깥 validate_outputs의 checks 목록을 갱신합니다. 별도 반환값은 없습니다.
    def check(name, passed, detail):
        checks.append(dict(name=name, passed=bool(passed), detail=detail))
    total_meetings = pq.ParquetFile(out / 'meetings.parquet').metadata.num_rows
    check('입력 회의 수 보존', total_meetings == expected_meetings, total_meetings)
    check('회의 ID 유일성', len(metadata) == total_meetings, len(metadata))
    keywords = list(iter_parquet(out / 'meeting_keywords.parquet'))
    keys = {(r['meeting_id'], r['keyword']): r for r in keywords}
    check('회의-키워드 유일성', len(keys) == len(keywords), len(keywords))
    grouped = defaultdict(list)
    refs_ok = counts_ok = scores_ok = True
    for r in keywords:
        mid = r['meeting_id']
        grouped[mid].append(r['rank'])
        refs_ok &= mid in metadata and all(r[key] == metadata[mid][key] for key, _ in COMMON)
        counts_ok &= r['frequency'] > 0 and 0 < r['document_frequency'] <= total_meetings
        scores_ok &= math.isfinite(r['score']) and 0 < r['score'] <= 1.0000001
    check('키워드 회의·원본 메타정보 연결', refs_ok, len(keywords))
    check('키워드 빈도와 문서빈도', counts_ok, len(keywords))
    check('정규화 점수 범위', scores_ok, len(keywords))
    check('순위 연속성과 상위 개수', all(sorted(ranks) == list(range(1, len(ranks) + 1)) and len(ranks) <= top_k for ranks in grouped.values()), len(grouped))
    exclusions = json.loads((out / 'exclusion_lists.json').read_text(encoding='utf-8'))
    excluded_names = {council: set(names) for council, names in exclusions['speaker_names_by_council'].items()}
    excluded_roles = set(exclusions['speaker_roles'])
    # 기능: 저장된 화자·직책 제외 목록을 이용해 후보 적격성을 검증 단계에서 다시 판단합니다.
    # 입력: 후보 문자열 word와 회의 ID mid. 회의 메타정보와 제외 목록은 바깥 함수에서 읽습니다.
    # 반환: 유효 후보이면 True. 이 결과로 빈도·TF-IDF·상위 순위·이력 범위를 다시 계산합니다.
    def valid_candidate(word, mid):
        council = metadata[mid]['council']
        names = excluded_names.get(council, set())
        return (word not in STOP_PHRASES and word.replace(' ', '') not in excluded_roles
                and word not in {council, council.removesuffix('의회')}
                and not any(part in STOPWORDS or part in names for part in word.split()))
    recomputed_df = Counter()
    for row in iter_parquet(out / 'term_counts.parquet'):
        recomputed_df.update(word for word in row['terms'] if valid_candidate(word, row['meeting_id']))
    vocabulary = {r['keyword'] for r in keywords}
    by_meeting_keywords = defaultdict(list)
    for row in keywords:
        by_meeting_keywords[row['meeting_id']].append(row)
    history_iterator = iter(iter_parquet(out / 'keyword_history.parquet', batch_size=4096))
    pending = next(history_iterator, None)
    frequency_match = coverage_match = tfidf_match = rank_match = True
    for row in iter_parquet(out / 'term_counts.parquet'):
        mid = row['meeting_id']
        raw_counts = dict(zip(row['terms'], row['counts']))
        filtered = {word: count for word, count in raw_counts.items() if valid_candidate(word, mid)}
        weights = {word: (1 + math.log(count)) * (math.log((1 + total_meetings) / (1 + recomputed_df[word])) + 1)
                   for word, count in filtered.items()}
        denominator = math.sqrt(sum(value * value for value in weights.values())) or 1.0
        expected_scores = {word: value / denominator for word, value in weights.items()}
        for kw in by_meeting_keywords[mid]:
            frequency_match &= kw['frequency'] == raw_counts.get(kw['keyword'])
            frequency_match &= kw['document_frequency'] == recomputed_df[kw['keyword']]
            tfidf_match &= math.isclose(kw['score'], expected_scores.get(kw['keyword'], 0), rel_tol=1e-12)
        expected_top = []
        for word, _ in sorted(expected_scores.items(), key=lambda item: (-item[1], item[0])):
            if any((' ' + word + ' ') in (' ' + chosen + ' ') for chosen in expected_top): continue
            expected_top.append(word)
            if len(expected_top) == top_k: break
        actual_top = [kw['keyword'] for kw in sorted(by_meeting_keywords[mid], key=lambda kw: kw['rank'])]
        rank_match &= actual_top == expected_top
        actual_history = {}
        while pending is not None and pending['meeting_id'] == mid:
            word = pending['keyword']
            coverage_match &= word not in actual_history
            actual_history[word] = pending
            pending = next(history_iterator, None)
        coverage_match &= set(actual_history) == vocabulary.intersection(filtered)
        for word, history_row in actual_history.items():
            frequency_match &= history_row['frequency'] == raw_counts.get(word)
            tfidf_match &= math.isclose(history_row['score'], expected_scores.get(word, 0), rel_tol=1e-12)
    coverage_match &= pending is None
    check('후보 빈도·문서빈도 재대조', frequency_match, len(keywords))
    check('TF-IDF 점수 재계산', tfidf_match, len(keywords))
    check('상위 키워드 선정 재계산', rank_match, total_meetings)
    check('비상위 출현까지 전체 키워드 이력 보존', coverage_match, len(vocabulary))
    history_ok = True
    history_top = set()
    catalog_counts = defaultdict(lambda: [0, 0])
    last_mid = None
    seen_words = set()
    for r in iter_parquet(out / 'keyword_history.parquet', batch_size=4096):
        mid, word = r['meeting_id'], r['keyword']
        if mid != last_mid:
            last_mid = mid
            seen_words.clear()
        history_ok &= word not in seen_words
        seen_words.add(word)
        top = keys.get((mid, word))
        history_ok &= mid in metadata and all(r[key] == metadata[mid][key] for key, _ in COMMON)
        history_ok &= r['frequency'] > 0 and r['keyword_id'] == word
        history_ok &= r['is_top_keyword'] == (top is not None)
        history_ok &= r['keyword_rank'] == (top['rank'] if top else None)
        if top:
            history_top.add((mid, word))
            history_ok &= r['frequency'] == top['frequency'] and math.isclose(r['score'], top['score'], rel_tol=1e-12)
        catalog_counts[word][0] += 1
        catalog_counts[word][1] += r['frequency']
    check('이력 연결·빈도·상위 여부', history_ok, len(catalog_counts))
    check('상위 키워드의 이력 보존', history_top == set(keys), len(history_top))
    catalog = list(iter_parquet(out / 'keyword_catalog.parquet'))
    check('키워드 카탈로그 집계', len(catalog) == len(catalog_counts) and all(catalog_counts[r['keyword']] == [r['meeting_count'], r['total_frequency']] for r in catalog), len(catalog))
    evidence_ok = True
    evidence_seen = set()
    for r in iter_parquet(out / 'keyword_evidence.parquet'):
        evidence_ok &= (r['meeting_id'], r['keyword']) in keys and 1 <= r['evidence_rank'] <= 3
        evidence_ok &= r['source_end'] - r['source_start'] == len(r['sentence_raw'])
        evidence_seen.add((r['meeting_id'], r['keyword']))
    check('키워드 근거 문장 연결', evidence_ok and evidence_seen == set(keys), len(evidence_seen))
    result = dict(passed=all(r['passed'] for r in checks), checks=checks)
    write_json(out / 'validation.json', result)
    if not result['passed']:
        raise AssertionError('검증 실패: validation.json을 확인하세요.')
    return result


# 기능: 경로·옵션 점검 후 전처리→키워드·이력 추출→무결성 검증을 순서대로 실행합니다.
# 입력: 로컬 입력·출력 폴더, 상위 개수·작업 수·파일별 표본 수. 출력은 입력과 다르고 비어 있어야 합니다.
# 저장: 결과 Parquet와 JSON. 처리 시작 표시를 만들고, 정상 완료 시 실행 명세를 기록한 뒤 시작 표시를 지웁니다.
# 반환: 실행 통계 사전. 처리 단계 예외는 FAILED.json에 기록 후 다시 발생시키며 HTML은 뒤쪽 별도 셀에서 만듭니다.
def run_local_extraction(input_dir, output_dir, top_k=20, workers=4, limit_per_file=0):
    input_dir, out = Path(input_dir), Path(output_dir)
    if not out.is_absolute():
        raise ValueError('OUTPUT_DIR에는 로컬 절대 경로를 입력하세요.')
    if input_dir.resolve() == out.resolve():
        raise ValueError('원본 입력 폴더와 출력 폴더는 달라야 합니다.')
    if top_k < 1 or workers < 1 or limit_per_file < 0:
        raise ValueError('추출 개수·작업 수는 양수이고 표본 수는 0 이상이어야 합니다.')
    if out.exists() and any(out.iterdir()):
        raise FileExistsError('출력 폴더에 파일이 있습니다. 비어 있는 새 출력 경로를 지정하세요.')
    inventory = inspect_inputs(input_dir)
    expected = sum(min(item['rows'], limit_per_file) if limit_per_file else item['rows'] for item in inventory)
    out.mkdir(parents=True, exist_ok=True)
    write_json(out / 'RUNNING.json', {'status': 'running', 'expected_meetings': expected})
    manifest = dict(input_dir=str(input_dir.absolute()), output_dir=str(out.absolute()), inputs=inventory,
                    parameters=dict(top_keywords=top_k, workers=workers, limit_per_file=limit_per_file, max_ngram=MAX_NGRAM),
                    metadata_columns=dict(meeting_name=MEETING_NAME_COLUMNS, council=COUNCIL_COLUMNS, date=DATE_COLUMNS),
                    stopwords=sorted(STOPWORDS), stop_phrases=sorted(STOP_PHRASES),
                    packages={p: importlib.metadata.version(p) for p in ('pyarrow', 'kiwipiepy', 'kiwipiepy_model')},
                    method='Kiwi 명사/연속 명사구 + 모든 입력 회의의 통합 TF-IDF',
                    id_policy='회의: 입력 파일 절대 경로 + 0-based 물리 행 번호; 키워드: NFC 정규화 문자열')
    try:
        meetings, people, roles, audit = preprocess(inventory, out, workers, limit_per_file)
        stats = extract_keywords(out, meetings, people, roles, top_k)
        validation = validate_outputs(out, expected, top_k)
        manifest.update(status='completed', totals=stats, preprocessing_audit=audit,
                        validation_passed=validation['passed'])
        write_json(out / 'run_manifest.json', manifest)
        (out / 'RUNNING.json').unlink()
        print('추출 완료:', json.dumps(stats, ensure_ascii=False), flush=True)
        return stats
    except BaseException as error:
        write_json(out / 'FAILED.json', {'status': 'failed', 'error': str(error), 'type': type(error).__name__})
        raise


# 기능: 저장된 키워드 이력에서 정규화한 검색어와 정확히 일치하는 회의를 조회합니다.
# 입력: 결과 폴더, 검색 키워드, 선택적 회의 날짜 기준 연도(year=None이면 전체).
# 반환: 날짜·의회·회의 ID 순으로 정렬한 행 목록. 날짜 미상은 뒤에 배치하며 동의어 검색은 하지 않습니다.
def keyword_history(output_dir, keyword, year=None):
    keyword = re.sub(r'\s+', ' ', normalize(keyword).lower()).strip()
    dataset = ds.dataset(Path(output_dir) / 'keyword_history.parquet', format='parquet')
    condition = ds.field('keyword') == keyword
    if year is not None:
        condition = condition & (ds.field('meeting_year') == int(year))
    rows = dataset.to_table(filter=condition).to_pylist()
    return sorted(rows, key=lambda r: (r['meeting_date'] or '9999', r['council'], r['meeting_id']))


## 입력 확인 및 실행

이 시점부터 위에 지정한 로컬 입력 폴더를 읽습니다. 파일과 행 수를 표시한 다음 전체 분석을 실행합니다. 본문이 없거나 추출 후보가 없는 회의도 `meetings.parquet`에 남습니다. 실행 도중 오류가 나면 출력 폴더의 `FAILED.json`을 확인하고, 다시 실행할 때 새 출력 경로를 지정하세요.

**아래 첫 코드 셀:** `table_html`·`show_rows` 표시 함수를 정의하고 `inspect_inputs(INPUT_DIR)`로 파일 목록·행 수·크기를 확인합니다. 원문을 정제하거나 결과 Parquet를 만드는 단계는 아직 시작하지 않습니다.


In [ ]:
# 기능: 행 목록을 가로 스크롤 가능한 HTML 표로 만들고 셀 내용을 이스케이프합니다.
# 입력: 딕셔너리 행 목록과 표시할 열 순서. 열을 생략하면 첫 행의 키를 사용합니다.
# 반환: HTML 문자열. 빈 목록은 안내 문구를 반환하며 파일 저장은 하지 않습니다.
def table_html(rows, columns=None):
    if not rows:
        return '<p>표시할 항목이 없습니다.</p>'
    columns = columns or list(rows[0])
    head = ''.join('<th>' + html.escape(str(c)) + '</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(
        '<td>' + html.escape(str(row.get(c, '') if row.get(c) is not None else '')) + '</td>'
        for c in columns) + '</tr>' for row in rows)
    return '<div style="overflow:auto"><table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table></div>'

# 기능: 노트북 화면에 행 목록을 HTML 표로 보여줍니다.
# 입력: 표시할 행 목록과 선택 열 목록.
# 효과: IPython이 있으면 표를 표시하고 없으면 JSON으로 출력합니다. 저장된 결과 파일은 변경하지 않습니다.
def show_rows(rows, columns=None):
    try:
        from IPython.display import HTML, display
        display(HTML(table_html(rows, columns)))
    except ImportError:
        print(json.dumps(rows, ensure_ascii=False, indent=2, default=str))

inventory = inspect_inputs(INPUT_DIR)
print(f"입력 파일 {len(inventory):,}개 / 전체 {sum(r['rows'] for r in inventory):,}개 회의")
show_rows([{'파일': Path(r['path']).name, '회의 수': r['rows'],
            '크기(MB)': round(r['size_bytes'] / 1024**2, 2)} for r in inventory])


### 전체 추출과 검증 실행

**입력:** 앞에서 설정한 입력·출력 경로, 상위 키워드 수, 작업 수, 표본 수.

**처리:** `run_local_extraction`을 호출하여 전처리·키워드 선정·전체 이력 저장·무결성 검사를 순차 실행합니다. 이전 셀에서 정의한 함수들을 실제로 호출하는 구간입니다.

**결과:** 출력 폴더에 Parquet와 JSON이 생성되고, `stats`에 실행 통계가 저장됩니다. 실패하면 다음 미리보기로 진행하기 전에 오류 메시지와 출력 폴더의 상태 파일을 확인하세요.


In [ ]:
stats = run_local_extraction(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    top_k=TOP_KEYWORDS,
    workers=WORKERS,
    limit_per_file=LIMIT_PER_FILE,
)


## 결과 확인

| 결과 파일 | 내용 |
|---|---|
| `meeting_keywords.parquet` | 회의별 상위 키워드, 순위, 빈도, TF-IDF 점수, 회의명·날짜·출처 |
| `keyword_history.parquet` | 추적 대상 키워드의 회의별 출현 이력. `is_top_keyword=False`인 출현 포함 |
| `keyword_catalog.parquet` | 키워드별 회의 수·총빈도·최초/최종 일자·출현 의회 |
| `keyword_evidence.parquet` | 회의별 상위 키워드마다 먼저 등장한 근거 문장 최대 3개와 발언자·원문 좌표 |
| `meetings.parquet` | 모든 회의의 메타데이터·정제 통계·경고·분리한 머리말/꼬리말 |
| `sentences.parquet` | 보존/제외 문장, 발언자, 원문/정제문, 제외 이유, 후보어 |
| `term_counts.parquet` | 회의별 후보어와 원빈도. 재검증과 후속 분석용 |

`run_manifest.json`에는 입력 파일의 경로·크기·수정 시각, 실행 설정, 패키지 버전을 남깁니다. `validation.json`에는 무결성 검사, `exclusion_lists.json`에는 발언자 이름·직책 제외 목록을 저장합니다. 아래 셀은 같은 로컬 출력 폴더에 요약 HTML 보고서도 만듭니다.

**아래 코드 셀:** `preview_parquet`로 `meeting_keywords.parquet`의 첫 배치에서 최대 30행을 읽어 주요 열을 표시합니다. 전체 본문을 다시 읽거나 저장된 키워드를 수정하지 않습니다.


In [ ]:
# 기능: Parquet의 첫 배치에서 지정한 개수만 읽어 미리보기 데이터를 만듭니다.
# 입력: 파일 경로, 최대 미리보기 행 수 count(기본 30), 선택 열 목록.
# 반환: 최대 count개의 행 딕셔너리 목록. 파일에 행이 없으면 빈 목록입니다.
def preview_parquet(path, count=30, columns=None):
    with pq.ParquetFile(path) as file:
        batch = next(file.iter_batches(batch_size=count, columns=columns), None)
    return batch.to_pylist() if batch is not None else []

preview_columns = ['council', 'meeting_name', 'meeting_date', 'keyword',
                   'rank', 'frequency', 'score', 'input_file', 'input_row_index']
preview = preview_parquet(OUTPUT_DIR / 'meeting_keywords.parquet', columns=preview_columns)
show_rows(preview, preview_columns)
print('결과 폴더:', OUTPUT_DIR)


## 같은 키워드의 회의 이력 조회

기본 검색어는 `청년 일자리`입니다. 다른 단어를 보고 싶으면 `SEARCH_KEYWORD`만 바꾸세요. 정확히 같은 정규화 키워드를 조회하며 동의어를 자동 병합하지 않습니다. 결과가 없다면 `keyword_catalog.parquet`의 실제 키워드를 확인하세요. `SEARCH_YEAR = None`은 연도 제한 없음이며, `2024`나 `2025`를 입력하면 **회의 날짜 기준**으로 제한합니다. 날짜가 누락된 회의는 연도 제한 시 제외됩니다. 파일명 기준 연도는 `source_year`에 별도 보존됩니다.

**아래 코드 셀:** `SEARCH_KEYWORD`와 `SEARCH_YEAR`를 `keyword_history`에 전달하고, 일치하는 이력의 총건수와 최대 100행을 화면에 표시합니다. 반환되는 `history_rows`에는 일치하는 전체 이력이 들어 있습니다.


In [ ]:
SEARCH_KEYWORD = '청년 일자리'
SEARCH_YEAR = None

history_rows = keyword_history(OUTPUT_DIR, SEARCH_KEYWORD, SEARCH_YEAR)
print(f'{SEARCH_KEYWORD!r} 출현 회의: {len(history_rows):,}개 (화면에는 최대 100개)')
show_rows(history_rows[:100], ['meeting_date', 'council', 'meeting_name', 'keyword',
    'frequency', 'is_top_keyword', 'keyword_rank', 'input_file', 'input_row_index'])

# 이 조회는 저장된 전체 이력 파일을 변경하지 않습니다.


### 검색 키워드의 근거 문장 조회

**입력:** 바로 앞 셀의 검색어·연도와 `keyword_evidence.parquet`.

**처리:** PyArrow 필터로 같은 검색어·연도의 근거 문장을 찾아 최대 30행을 읽습니다. 근거 파일은 각 회의의 상위 키워드에 대해서만 문장을 저장합니다.

**결과:** 회의명·발언자·원문 문장·원본 파일 경로·행 번호·문자 좌표를 화면에 표시합니다. 이력에 출현하더라도 해당 회의의 상위 키워드가 아니면 이 표에 근거가 없을 수 있습니다.


In [ ]:
normalized_search = re.sub(r'\s+', ' ', normalize(SEARCH_KEYWORD).lower()).strip()
evidence_dataset = ds.dataset(OUTPUT_DIR / 'keyword_evidence.parquet', format='parquet')
evidence_filter = ds.field('keyword') == normalized_search
if SEARCH_YEAR is not None:
    evidence_filter = evidence_filter & (ds.field('meeting_year') == int(SEARCH_YEAR))
evidence_preview = evidence_dataset.scanner(filter=evidence_filter).head(30).to_pylist()
print('상위 키워드로 선정된 회의의 근거 문장 (최대 30행)')
show_rows(evidence_preview, ['meeting_name', 'meeting_date', 'keyword', 'speaker_name',
    'speaker_role', 'sentence_raw', 'input_file', 'input_row_index', 'source_start', 'source_end'])


## 로컬 HTML 요약 보고서

이 보고서에는 실행 규모, 의회·연도별 회의 수, 정제 경고, 출현 회의 수가 많은 키워드와 결과 미리보기를 담습니다. 전체 이력과 원문 근거는 Parquet 파일에 보존됩니다.

**입력 → 처리 → 결과:** 실행 명세·검증 JSON과 회의·키워드 Parquet → 의회/연도별 회의 수와 경고·상위 집계 취합 및 HTML 구성 → `OUTPUT_DIR/keyword_report.html` 저장. 표의 텍스트는 HTML 이스케이프 처리하며, 외부 서버 없이 브라우저에서 열 수 있습니다.


In [ ]:
import heapq

manifest = json.loads((OUTPUT_DIR / 'run_manifest.json').read_text(encoding='utf-8'))
validation = json.loads((OUTPUT_DIR / 'validation.json').read_text(encoding='utf-8'))
meeting_groups = Counter()
warning_counts = Counter()
for item in iter_parquet(OUTPUT_DIR / 'meetings.parquet'):
    meeting_groups[(item['council'], item['meeting_year'], item['source_year'])] += 1
    warning_counts.update(item['warnings'])
group_rows = [dict(council=k[0], meeting_year=k[1], source_year=k[2], meetings=v)
              for k, v in sorted(meeting_groups.items(), key=lambda kv: str(kv[0]))]
popular = heapq.nsmallest(30, iter_parquet(OUTPUT_DIR / 'keyword_catalog.parquet'),
                         key=lambda r: (-r['meeting_count'], -r['total_frequency'], r['keyword']))
sample_note = ('파일별 표본 실행입니다. 전체 데이터 결과가 아닙니다.'
               if LIMIT_PER_FILE else '설정된 입력 폴더의 전체 행을 처리했습니다.')
report = '<!doctype html><html lang="ko"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">'
report += '<title>지방의회 키워드 추출 보고서</title><style>'
report += 'body{font-family:system-ui,sans-serif;max-width:1200px;margin:32px auto;padding:0 20px;color:#172438;line-height:1.65}h1,h2{line-height:1.3}table{border-collapse:collapse;width:100%;font-size:14px}th,td{padding:9px;border:1px solid #dce3ed;text-align:left;vertical-align:top;overflow-wrap:anywhere;max-width:420px}th{background:#eef3f9}section{margin:32px 0}.note{padding:16px;background:#eef6ff;border-radius:8px}code{overflow-wrap:anywhere}</style><body>'
report += '<h1>지방의회 회의록 키워드 추출</h1><p class="note">' + html.escape(sample_note) + '</p>'
report += '<p>입력: <code>' + html.escape(str(INPUT_DIR)) + '</code><br>출력: <code>' + html.escape(str(OUTPUT_DIR)) + '</code></p>'
report += '<section><h2>처리 규모</h2>' + table_html([stats]) + '</section>'
report += '<section><h2>의회·연도별 회의 수</h2>' + table_html(group_rows) + '</section>'
report += '<section><h2>주의해서 확인할 회의</h2>' + table_html([{'경고': k, '회의 수': v} for k, v in warning_counts.most_common()]) + '</section>'
report += '<section><h2>출현 회의 수가 많은 키워드 30개</h2>' + table_html(popular) + '</section>'
report += '<section><h2>회의별 키워드 미리보기</h2>' + table_html(preview, preview_columns) + '</section>'
report += '<section><h2>무결성 검사</h2>' + table_html(validation['checks']) + '</section>'
report += '<section><h2>해석 범위</h2><p>Kiwi 명사·연속 명사구와 모든 입력 회의를 함께 계산한 TF-IDF로 추출했습니다. 점수는 중요성의 절대 척도나 확률이 아닙니다. 같은 키워드는 같은 표기를 뜻하며, 동일 정책이나 동의어를 자동 판별하지 않습니다. 발언자·의전 문장 판별은 규칙 기반이므로 실제 표본을 점검해야 합니다. 근거 문장은 먼저 등장한 최대 3개로, 추상적 요약문이 아닙니다. 토픽 모델링과 인물 네트워크는 수행하지 않았습니다.</p></section></body></html>'
report_path = OUTPUT_DIR / 'keyword_report.html'
report_path.write_text(report, encoding='utf-8')
print('HTML 보고서 저장:', report_path)
print('무결성 검사:', sum(c['passed'] for c in validation['checks']), '/', len(validation['checks']))


## 해석 시 확인할 점

- **점수:** 회의별 빈도에 로그를 적용하고 모든 입력 회의의 공통 IDF로 가중한 후 회의 단위로 정규화합니다. 확률이 아닙니다. 파일·연도를 추가하면 IDF와 순위가 바뀌므로 연도 비교는 두 연도를 함께 실행한 결과를 사용하세요.
- **정제:** 화자 표식과 직책·성명 패턴으로 발언자를 분리합니다. 첫 화자 앞 머리말과 최종 산회 뒤 꼬리말은 키워드 추출에서 제외하고 별도 보존합니다. 비표준 표식이나 이름은 인식하지 못할 수 있으며, 혼합 발언은 가능한 한 유지합니다. 추출 규칙만으로 모든 의전 표현이 사라지는 것은 아닙니다.
- **추적 범위:** 적어도 한 회의에서 상위 키워드로 선정된 단어가 추적 대상입니다. 그 단어가 불용어·발언자 이름 제외 규칙을 통과한 다른 회의의 출현을 모두 저장합니다. 전체 후보어는 `term_counts.parquet`에 별도로 있습니다.
- **근거:** 문장 위치는 정제 전 원문 기준이며 추출 시 직접 대조합니다. 근거 문장 최대 3개는 상위 키워드별 첫 출현 문장입니다. 요약 모델이나 의미상 유사도 모델은 사용하지 않았습니다.
- **메모리와 시간:** 원문은 파일 배치·회의 단위로 처리하고 문장은 순차 저장합니다. 전체 어휘의 문서빈도, 회의 메타정보와 회의별 상위 키워드는 메모리에 유지하므로 매우 큰 자료에서는 메모리가 더 필요할 수 있습니다. `LIMIT_PER_FILE=2`로 새 출력 폴더에 시험한 뒤, 전체 실행은 `0`과 다른 빈 출력 폴더를 사용하세요.
- **품질 확인:** 자동 무결성 검사는 연결·빈도·점수의 일관성을 확인합니다. 키워드의 의미적 적절성을 보장하지 않으므로 여러 지역의 실제 근거 문장을 확인하고 불용어를 조정하세요. 정당·성향 추정, 토픽 모델링, 네트워크 분석은 이 노트북에 포함하지 않습니다.
